# Lab 07 — Attention, Memory Traffic, and Tiling
**Goal:** Visualize why naive attention materializes a large attention matrix and understand the core idea behind FlashAttention-style tiling.

Free-first: run on CPU or free Colab where possible.

Method: **predict → run → inspect → break → decide**.


In [ ]:
import torch
def attention_memory(B,H,T,D,dtype_bytes=2):
    score_bytes=B*H*T*T*dtype_bytes
    qkv_bytes=3*B*H*T*D*dtype_bytes
    return score_bytes,qkv_bytes
for T in [512,1024,2048,4096]:
    s,q=attention_memory(1,8,T,64)
    print(T,"score GiB",s/1024**3,"QKV GiB",q/1024**3)


## Predict before running
Predict what happens to the attention-score tensor if sequence length doubles.


In [ ]:
import matplotlib.pyplot as plt
Ts=[256,512,1024,2048,4096]
vals=[attention_memory(1,8,T,64)[0]/1024**3 for T in Ts]
plt.plot(Ts,vals,marker="o")
plt.xlabel("sequence length")
plt.ylabel("attention-score memory (GiB, illustrative)")
plt.title("Naive attention score matrix scales quadratically")
plt.show()


## Explain the result
The score tensor has T² scaling. FlashAttention-style approaches reduce high-bandwidth-memory traffic and avoid materializing the full score matrix, using tiling and online normalization. The key idea is systems-level: same mathematical result, different memory behavior.


In [ ]:
# Thought experiment:
# T -> 2T means the score matrix has 4x as many entries.
# A tiled algorithm avoids materializing the full T x T matrix.
for T in [512,1024,2048,4096,8192]:
    print(T, "relative score entries:", (T/512)**2)


## Critical-thinking answers
Do not describe FlashAttention as “just a faster softmax.” It is a carefully engineered attention implementation. The training lesson is to connect algorithmic complexity to the hardware memory hierarchy.

### Sources
https://arxiv.org/abs/2205.14135
